In [5]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import warnings
import re
warnings.filterwarnings('ignore')
from datetime import datetime

In [6]:
# ==========================================
# 1. SETUP PATHS
# ==========================================

# ── FULL SCALE PATHS ──────────────────────────────────

# Full Philippines OSM shapefile folder
base_osm_dir = "/Users/ruben/Desktop/Thesis/TrainingData/PH_OSM.shp"

# Full DHS GPS shapefile (all 1247 clusters, same file as before)
dhs_shp_path = "/Users/ruben/Desktop/Thesis/TrainingData/PH_DHS_GPS/PHGE81FL/PHGE81FL.shp"

# Full VIIRS labels CSV (generated from viirs_ntl_labels_all_clusters.csv)
viirs_csv_path = "/Users/ruben/Desktop/Thesis/TrainingData/viirs_ntl_labels_all_clusters.csv"

# Output: full static features for all 1247 clusters
output_csv = "/Users/ruben/Desktop/Thesis/TrainingData/static_osm_features_full.csv"

In [7]:
# ==========================================
# 2. LOAD DATA
# ==========================================
print("Loading Shapefiles...")
gdf_dhs = gpd.read_file(dhs_shp_path)

# Load OSM files (Handle missing files gracefully)
try:
    gdf_roads = gpd.read_file(os.path.join(base_osm_dir, "gis_osm_roads_free_1.shp"))
    gdf_bldgs = gpd.read_file(os.path.join(base_osm_dir, "gis_osm_buildings_a_free_1.shp"))
    gdf_pois = gpd.read_file(os.path.join(base_osm_dir, "gis_osm_pois_free_1.shp"))
    print("Loaded Roads, Buildings, and POIs.")
except Exception as e:
    print(f"Error loading OSM files: {e}")
    print("Ensure you have roads, buildings, and POIs in the folder.")

Loading Shapefiles...
Loaded Roads, Buildings, and POIs.


In [8]:
# ==========================================
# 3. REPROJECT & BUFFER
# ==========================================
# Project to Meters (Philippines Zone 51N)
target_crs = "EPSG:32651"
gdf_dhs = gdf_dhs.to_crs(target_crs)
gdf_roads = gdf_roads.to_crs(target_crs)
gdf_bldgs = gdf_bldgs.to_crs(target_crs)
gdf_pois = gdf_pois.to_crs(target_crs)

# DYNAMIC BUFFERING (Tingzon Method)
# Check if DHS has 'URBAN_RURAL' column (usually 'URBAN_RURA' in shapefiles)
# U = Urban, R = Rural. If missing, default to 5km.
print("Creating Dynamic Buffers...")
def get_buffer(row):
    # Adjust column name if necessary (e.g., 'URBAN_RURA', 'TYPE')
    if 'URBAN_RURA' in row and row['URBAN_RURA'] == 'U':
        return row.geometry.buffer(2000) # 2km for Urban
    else:
        return row.geometry.buffer(5000) # 5km for Rural/Unknown

gdf_dhs['buffer_geom'] = gdf_dhs.apply(get_buffer, axis=1)

Creating Dynamic Buffers...


In [9]:
# ==========================================
# 4. FEATURE ENGINEERING LOOP
# ==========================================
print("Extracting Detailed Features (Roads, Buildings, POIs)...")
results = []

road_sindex = gdf_roads.sindex
bldg_sindex = gdf_bldgs.sindex
poi_sindex  = gdf_pois.sindex

for loop_idx, (idx, row) in enumerate(gdf_dhs.iterrows()):
    cluster_id = row['DHSCLUST']
    buffer     = row['buffer_geom']

    # ── ROADS ──────────────────────────────────────────
    possible_roads = gdf_roads.iloc[
        list(road_sindex.intersection(buffer.bounds))]
    precise_roads  = possible_roads[
        possible_roads.intersects(buffer)]

    road_types = {
        'Main_Roads'     : ['motorway', 'trunk',
                            'primary', 'primary_link'],
        'Secondary_Roads': ['secondary', 'tertiary'],
        'Local_Roads'    : ['residential', 'living_street',
                            'unclassified', 'service'],
        'Tracks'         : ['track', 'path']
    }

    road_stats = {}
    if len(precise_roads) > 0:
        clipped_geoms = precise_roads.geometry.intersection(buffer)
        road_stats['Total_Road_Length'] = (
            clipped_geoms.length.sum() / 1000.0)
        for r_cat, r_classes in road_types.items():
            mask = precise_roads['fclass'].isin(r_classes)
            road_stats[f'{r_cat}_Length'] = (
                clipped_geoms[mask].length.sum() / 1000.0)
    else:
        road_stats = {f'{k}_Length': 0
                      for k in road_types.keys()}
        road_stats['Total_Road_Length'] = 0

    # ── BUILDINGS ──────────────────────────────────────
    possible_bldgs = gdf_bldgs.iloc[
        list(bldg_sindex.intersection(buffer.bounds))]
    precise_bldgs  = possible_bldgs[
        possible_bldgs.intersects(buffer)]

    bldg_stats   = {'Total_Bldg_Count': len(precise_bldgs)}
    target_bldgs = ['residential', 'commercial',
                    'industrial', 'school', 'hospital']

    if len(precise_bldgs) > 0:
        type_col = ('type' if 'type' in precise_bldgs.columns
                    else 'fclass')
        counts = precise_bldgs[type_col].value_counts()
        for t in target_bldgs:
            bldg_stats[f'Bldg_{t}_Count'] = counts.get(t, 0)
        bldg_stats['Total_Bldg_Area'] = (
            precise_bldgs.area.sum())
    else:
        for t in target_bldgs:
            bldg_stats[f'Bldg_{t}_Count'] = 0
        bldg_stats['Total_Bldg_Area'] = 0

    # ── POIs ───────────────────────────────────────────
    possible_pois = gdf_pois.iloc[
        list(poi_sindex.intersection(buffer.bounds))]
    precise_pois  = possible_pois[
        possible_pois.intersects(buffer)]

    poi_stats  = {'Total_POI_Count': len(precise_pois)}
    wealth_pois = ['bank', 'hotel', 'fast_food',
                   'convenience', 'school', 'hospital']

    if len(precise_pois) > 0:
        counts = precise_pois['fclass'].value_counts()
        for p in wealth_pois:
            poi_stats[f'POI_{p}_Count'] = counts.get(p, 0)
    else:
        for p in wealth_pois:
            poi_stats[f'POI_{p}_Count'] = 0

    # ── COMBINE ────────────────────────────────────────
    feature_row = {'DHSCLUST': cluster_id}
    feature_row.update(road_stats)
    feature_row.update(bldg_stats)
    feature_row.update(poi_stats)
    results.append(feature_row)

    # Progress + checkpoint
    n_done = loop_idx + 1
    if n_done % 50 == 0 or n_done == len(gdf_dhs):
        print(f"  [{datetime.now().strftime('%H:%M:%S')}] "
              f"{n_done}/{len(gdf_dhs)} clusters done")
    if n_done % 200 == 0:
        cp = output_csv.replace('.csv',
                                f'_checkpoint_{n_done}.csv')
        pd.DataFrame(results).to_csv(cp, index=False)
        print(f"  Checkpoint saved: {cp}")

Extracting Detailed Features (Roads, Buildings, POIs)...
  [00:06:28] 50/1247 clusters done
  [00:06:29] 100/1247 clusters done
  [00:06:30] 150/1247 clusters done
  [00:06:30] 200/1247 clusters done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/TrainingData/static_osm_features_full_checkpoint_200.csv
  [00:06:30] 250/1247 clusters done
  [00:06:31] 300/1247 clusters done
  [00:06:35] 350/1247 clusters done
  [00:06:38] 400/1247 clusters done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/TrainingData/static_osm_features_full_checkpoint_400.csv
  [00:06:39] 450/1247 clusters done
  [00:06:40] 500/1247 clusters done
  [00:06:41] 550/1247 clusters done
  [00:06:42] 600/1247 clusters done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/TrainingData/static_osm_features_full_checkpoint_600.csv
  [00:06:44] 650/1247 clusters done
  [00:06:45] 700/1247 clusters done
  [00:06:45] 750/1247 clusters done
  [00:06:46] 800/1247 clusters done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/Train

In [10]:
# ==========================================
# 5. MERGE VIIRS + SAVE FULL DATASET
# ==========================================
import time
from datetime import datetime

# Build the full OSM dataframe from results
df_osm = pd.DataFrame(results)
df_osm['DHSCLUST'] = df_osm['DHSCLUST'].astype(int)

print(f"OSM features extracted: {len(df_osm)} clusters")
print(f"Columns: {list(df_osm.columns)}")

# Load VIIRS labels (full 1247 clusters)
print(f"\nLoading VIIRS data from: {viirs_csv_path}")
df_viirs = pd.read_csv(viirs_csv_path)

# Normalize column names
# Handle different possible column names from GEE export
col_map = {}
for col in df_viirs.columns:
    if col.lower() in ('cluster_id', 'clusterid'):
        col_map[col] = 'DHSCLUST'
    if col.lower() in ('ntl_value', 'median', 'avg_rad'):
        col_map[col] = 'VIIRS_Median'
if col_map:
    df_viirs = df_viirs.rename(columns=col_map)

# Keep only what we need
if 'DHSCLUST' in df_viirs.columns and 'VIIRS_Median' in df_viirs.columns:
    df_viirs_clean = df_viirs[['DHSCLUST', 'VIIRS_Median']].copy()
elif 'DHSCLUST' in df_viirs.columns and 'NTL_Value' in df_viirs.columns:
    df_viirs_clean = df_viirs[['DHSCLUST', 'NTL_Value']].copy()
    df_viirs_clean = df_viirs_clean.rename(columns={'NTL_Value': 'VIIRS_Median'})
else:
    print(f"WARNING: Could not find VIIRS value column.")
    print(f"Available columns: {list(df_viirs.columns)}")
    # Create empty VIIRS column as fallback
    df_viirs_clean = pd.DataFrame({
        'DHSCLUST'    : df_osm['DHSCLUST'],
        'VIIRS_Median': 0.0
    })

df_viirs_clean['DHSCLUST'] = df_viirs_clean['DHSCLUST'].astype(int)

# Check coverage before merge
viirs_ids = set(df_viirs_clean['DHSCLUST'])
osm_ids   = set(df_osm['DHSCLUST'])
missing_viirs = osm_ids - viirs_ids
if missing_viirs:
    print(f"\nWARNING: {len(missing_viirs)} clusters have no VIIRS data.")
    print(f"  These will get VIIRS_Median = 0 after merge.")
    print(f"  Sample missing: {sorted(list(missing_viirs))[:10]}")

# Merge OSM + VIIRS
df_final = pd.merge(
    df_osm,
    df_viirs_clean,
    on    = 'DHSCLUST',
    how   = 'left'
)

# Fill missing VIIRS with 0
df_final['VIIRS_Median'] = df_final['VIIRS_Median'].fillna(0)

# Sort by cluster ID
df_final = df_final.sort_values('DHSCLUST').reset_index(drop=True)

# Save
df_final.to_csv(output_csv, index=False)

print("\n" + "="*50)
print("SUCCESS")
print("="*50)
print(f"Clusters          : {len(df_final)}")
print(f"Columns           : {len(df_final.columns)}")
print(f"Expected columns  : ~21 (20 OSM + VIIRS_Median)")
print(f"VIIRS zeros       : {(df_final['VIIRS_Median'] == 0).sum()}")
print(f"Saved to          : {output_csv}")
print(f"\nSample:")
print(df_final.head())
print(f"\nColumn list:")
print(list(df_final.columns))

OSM features extracted: 1247 clusters
Columns: ['DHSCLUST', 'Total_Road_Length', 'Main_Roads_Length', 'Secondary_Roads_Length', 'Local_Roads_Length', 'Tracks_Length', 'Total_Bldg_Count', 'Bldg_residential_Count', 'Bldg_commercial_Count', 'Bldg_industrial_Count', 'Bldg_school_Count', 'Bldg_hospital_Count', 'Total_Bldg_Area', 'Total_POI_Count', 'POI_bank_Count', 'POI_hotel_Count', 'POI_fast_food_Count', 'POI_convenience_Count', 'POI_school_Count', 'POI_hospital_Count']

Loading VIIRS data from: /Users/ruben/Desktop/Thesis/TrainingData/viirs_ntl_labels_all_clusters.csv

SUCCESS
Clusters          : 1247
Columns           : 21
Expected columns  : ~21 (20 OSM + VIIRS_Median)
VIIRS zeros       : 0
Saved to          : /Users/ruben/Desktop/Thesis/TrainingData/static_osm_features_full.csv

Sample:
   DHSCLUST  Total_Road_Length  Main_Roads_Length  Secondary_Roads_Length  \
0         1          54.771597           1.804234                6.857690   
1         2          43.582598           5.2157